In [19]:
class Intersection:
    def __init__(self, green_light, red_light, yellow_light=3, capacity_ver=2.0, capacity_hor=2.0, offset = 0.0):
        self.green_light = green_light
        self.red_light = red_light
        self.yellow_light = yellow_light
        self.capacity_ver = capacity_ver
        self.capacity_hor = capacity_hor
        
        self.queue_N = 0
        self.queue_S = 0
        self.queue_W = 0
        self.queue_E = 0        

        self.temp_timer = 0

        # STATES
        # 0 - green vertical
        # 1 - yellow - all stop
        # 2 - green horizontal
        # 3 - yellow - all stop
        self.state = 0
        self.state_times = (self.green_light, self.yellow_light, self.red_light, self.yellow_light)
        offset_seconds = int(sum(self.state_times) * offset)
        for second in range(offset_seconds):
            self.temp_timer += 1
            if self.temp_timer > self.state_times[self.state]:
                self.state = (self.state + 1) % 4
                self.temp_timer = 1

    def step(self, input_N, input_S, input_W, input_E):
        outputs = {'queue_time_sum': 0, 'output_N': 0, 'output_S': 0, 'output_W': 0, 'output_E': 0}
        self.temp_timer += 1

        if self.temp_timer > self.state_times[self.state]:
            self.state = (self.state + 1) % 4
            self.temp_timer = 1

        if self.state == 0: # green vertical
            # vertical outputs
            outputs['output_N'] = min(self.queue_S + input_S, self.capacity_ver)
            outputs['output_S'] = min(self.queue_N + input_N, self.capacity_ver)

            # vertical drive
            self.queue_N = max(0, self.queue_N + input_N - self.capacity_ver)
            self.queue_S = max(0, self.queue_S + input_S - self.capacity_ver)

            # horizontal queue
            self.queue_W = self.queue_W + input_W
            self.queue_E = self.queue_E + input_E
        elif self.state == 2: # green horizontal
            # horizontal outputs
            outputs['output_W'] = min(self.queue_E + input_E, self.capacity_hor)
            outputs['output_E'] = min(self.queue_W + input_W, self.capacity_hor)

            # horizontal drive
            self.queue_W = max(0, self.queue_W + input_W - self.capacity_hor)
            self.queue_E = max(0, self.queue_E + input_E - self.capacity_hor)

            # vertical queue
            self.queue_N = self.queue_N + input_N
            self.queue_S = self.queue_S + input_S
        else: # yellow
            # horizontal queue
            self.queue_W = self.queue_W + input_W
            self.queue_E = self.queue_E + input_E
            # vertical queue
            self.queue_N = self.queue_N + input_N
            self.queue_S = self.queue_S + input_S

        queue_time_sum = self.queue_N + self.queue_S + self.queue_W + self.queue_E
        outputs['queue_time_sum'] = queue_time_sum
        outputs['state'] = self.state
        return outputs 

In [29]:
def simulation(params):

    keys = ('A', 'B', 'C', 'D')
    intersections = {}

    for i, key in enumerate(keys):
        g, r, o = params[i*3 : (i+1)*3]
        intersections[key] = Intersection(g, r, offset=o)

    connections = {
        ('A', 'output_E'): ('B', 'input_W'),
        ('B', 'output_W'): ('A', 'input_E'),
        ('B', 'output_E'): ('C', 'input_W'),
        ('C', 'output_W'): ('B', 'input_E'),
        ('B', 'output_S'): ('D', 'input_N'),
        ('D', 'output_N'): ('B', 'input_S')
    }

    last_outputs = {
        key: {'queue_time_sum': 0, 'output_N': 0, 'output_S': 0, 'output_W': 0, 'output_E': 0}
        for key in intersections.keys()
    }

    traffic_map = {
        'A': {'input_N': 0.5, 'input_S': 0.6, 'input_W': 1.5, 'input_E': 1.0},
        'B': {'input_N': 1.7, 'input_S': 1.0, 'input_W': 1.0, 'input_E': 1.0},
        'C': {'input_N': 0.4, 'input_S': 0.7, 'input_W': 1.0, 'input_E': 1.0},
        'D': {'input_N': 1.0, 'input_S': 0.5, 'input_W': 1.1, 'input_E': 0.3}
    }

    cost = 0

    for i in range(120):
        current_inputs = {
            key: traffic_map[key].copy()
            for key in intersections.keys()
        }
        
        for output, input in connections.items():
            intersection_out, road_out = output
            intersection_in, road_in = input
            current_inputs[intersection_in][road_in] = last_outputs[intersection_out][road_out]
            
        for id, intersection in intersections.items():
            last_outputs[id] = intersection.step(**current_inputs[id])
            cost = cost + last_outputs[id]['queue_time_sum']

    return(cost)

In [33]:
# intersections = {
#     'A': Intersection(10, 20, 0.2),
#     'B': Intersection(15, 10, 0),
#     'C': Intersection(10, 10, 0),
#     'D': Intersection(12, 8, 0.5)   
# }

output_from_metaheuristic = [10, 20, 0.2, 15, 10, 0, 10, 10, 0, 12, 8, 0.5]
result = simulation(output_from_metaheuristic)
print(result)

23838.3
